# Візуальна одометрія

Постановка задачі: за заданим відео оцінити обертання та поступальне переміщення камери між послідовними кадрами.

Дано:

   - Кожен піксель представлений у вигляді:

        $p=\begin{bmatrix} u \\ v \\ 1 \end{bmatrix}$

   - Камеру відкалібровано та матриця внутрішніх параметрів відома:

        $K=\begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$

   - Відомо щонайменше 5 пар відповідних ключових точок між кадром 1 і кадром 2:

        $p_{1,i} \leftrightarrow p_{2,i}, i=\overline{1,5}$

Перетворюємо кожен піксель на нормалізований промінь камери:

   $x=K^{-1}p$

## Суттєва матриця
Суттєва матриця — описує відносне обертання та поступальне переміщення між двома положеннями камери.
Перетворення координат точки реального світу $P$ із системи координат камери 1 у систему координат камери 2 описується рівнянням:

   $P_2 = RP_1 + t$

Тоді суттєва матриця визначається як:

   $E=[t]_{\times}R$

Де $[t]_{\times}$ — антисиметрична матриця, що відповідає вектору переміщення і має вигляд:

   $[t]_{\times}=\begin{bmatrix} 0 & -t_z & t_y \\ t_z & 0 & -t_x \\ -t_y & t_x & 0 \end{bmatrix}$

   $R$ — матриця обертання розміром 3x3

Суттєва матриця має задовольняти обмеженню:

   $x_2^TEx_1=0$

Рівняння випливає з того факту, що три вектори компланарні, якщо їхній мішаний добуток дорівнює нулю. Копланарність векторів $x_1, x_2, t$ випливає з трикутника епіполярної геометрії.

Матриця $E$ має 5 ступенів свободи (3 для обертання та 2 для переміщення).
Проте, ми не можемо просто розв’язати п’ять лінійних рівнянь для $E$, оскільки необхідно забезпечити, щоб сингулярні значення матриці мали вигляд:
    $(\sigma, \sigma, 0)$.

Для обчислення суттєвої матриці використовується метод п’яти точок. Метод п’яти точок забезпечує виконання обох обмежень, і зводить задачу до розв’язку поліноміального рівняння 10-го степеня.

На практиці, обчислення матриці може бути виконано за допомогою функції `cv2.findEssentialMat` з бібліотеки OpenCV.
Також на практиці при обчисленні матриці використовується метод RANSAC для підвищення стійкості оцінювання, але для цього методу потрібно більше ніж 5 пар точок.

## Матриця обертання $R$ та вектор переміщення $t$

Наступним кроком із суттєвої матриці $E$ необхідно отримати матрицю $R$ та вектор $t$. Для цього використовується сингулярний розклад (SVD):

   $E=U\Sigma V^T$

Для ідеальної суттєвої матриці сингулярні значення мають особливу структуру:

   $\Sigma=\begin{bmatrix} \sigma & 0 & 0 \\ 0 & \sigma & 0 \\ 0 & 0 & 0 \end{bmatrix}$

Оскільки сама матриця $E$ має довільний масштаб, нормалізуємо цей вираз до:

   $\Sigma=\begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 0 \end{bmatrix}$

Далі виділяємо шукані компоненти. Третій стовпець матриці $U$ формує кандидатів для вектора переміщення: $t=\pm u_3$.

Для знаходження матриці $R$ будуємо спеціальну матрицю:

   $W=\begin{bmatrix} 0 & -1 & 0 \\ 1 & 0 & 0 \\ 0 & 0 & 1 \end{bmatrix}$

Отримати допустимі матриці $R$ можна за формулами:

   $R_1=UWV^T$

   $R_2=UW^TV^T$

Далі треба виконати перевірку хіральності, щоб визначити фізично коректні варіанти $(R,t)$.
Саме по собі епіполярне обмеження не визначає однозначно, яка конфігурація є фізично коректною.
Отже, після сингулярного розкладу отримуємо: $candidate R_1, candidate R_2, candidate t, candidate -t$

Для кожного можливого варіанта $(R,t)$ потрібно перевірити, чи є він фізично коректним:
1. Тріангулювати кілька відповідних точок.
2. Перевірити їхні глибини.
3. Вибрати положення і орієнтацію, за яких точки розташовані перед обома камерами.

 На практиці, функція recoverPose() з OpenCV використовується для декомпозиції матриці $E$ і застосовує даний підхід для вибору фізично коректного відносного положення й орієнтації.

## Властивості вектору переміщення.

Властивість: кожен вектор переміщення має одиничну довжину.

З вектором переміщення пов’язані дві окремі проблеми:
1. Неоднозначність абсолютного масштабу: монокулярна візуальна одометрія не може відновити реальну пройдену відстань лише за двома зображеннями.
2. Узгодженість відносного масштабу: переміщення, оцінені за різними парами кадрів, можуть бути неузгодженими за масштабом, оскільки вектор переміщення завжди має одиничну довжину, незалежно від реально пройденої відстані. Алгоритм, що використовує кілька кадрів, може розв’язати цю проблему.

## Перетворення координат точки.

Нехай задано деяку точку реального світу $P$. Необхідно знайти її координати $P_0, P_1, ..., P_n$ у системі координат камери $0, 1,...,n$.

Просте додавання векторів переміщення є некоректним.
Перед додаванням наступного переміщення необхідно повернути попередній вектор переміщення в новішу систему координат.

Коректний підхід до отримання координат точки $P$ у системі координат $n-th$ камери:
   1. Обчислити попарні перетворення. Ці відносні перетворення послідовно об’єднуються для отримання траєкторії:

      $Frame \ 0 \rightarrow Frame \ 1: (R_{01}, t_{01})$

      $Frame \ 1 \rightarrow Frame \ 2: (R_{12}, t_{12})$
   2. Перетворення координат точки з однієї системи координат у наступну:

      $P_1=R_{01}P_0+t_{01}$

      $P_2=R_{12}P_1+t_{12}$
   3. Підставляємо друге рівняння у перше:

        $P_2=R_{12}(R_{01}P_0+t_{01})+t_{12} = R_{12}R_{01}P_0+R_{12}t_{01}+t_{12} \implies$

        $R_{02}=R_{12}R_{01}$

        $t_{02}=R_{12}t_{01}+t_{12}$

## Побудова траєкторії руху камери

Побудуємо траєкторію руху камери в системі координат камери $0$.

Позначення:
- $k$ — індекс системи координат камери

- $w$ — система координат реального світу

Алгоритм:

1. Перетворення координат точки із системи координат $i$ у систему $j$ можна подати матрицею однорідного перетворення:

    $T_{i,j} = \begin{bmatrix} R_{i, j} & t_{i, j} \\ 0 & 1 \end{bmatrix}$

    А сам процес перетворення виконується як множення матриць:

    $\begin{bmatrix} P_1 \\ 1 \end{bmatrix} = T_{0,1}\begin{bmatrix} P_0 \\ 1 \end{bmatrix}$

2. Оберемо систему координат камери 0 як систему координат реального світу. Її положення відносно самої себе визначається одиничною матрицею:

    $T_{0, w} = I_4 = \begin{bmatrix} I_3 & 0 \\ 0 & 1 \end{bmatrix}$

3. Знайдемо матрицю однорідного перетворення $T_{k, w}$, що переводить точку із системи координат камери $k$ у систему координат світу $w$:

    $P_w = T_{0,w}P_0=T_{1,w}P_1=T_{1,w}T_{0,1}P_0$ – оскільки $P$ — це одна й та сама точка, то після переходу з різних систем координат у $w$ вона повинна мати однакові координати.

    Тоді:

    $T_{1,w} = T_{0,w} T^{-1}_{0,1}$

    А у загальному випадку:

    $T_{k,w} = T_{k-1, w}T^{-1}_{k-1, k}$

    Обернена матриця:

    $T^{-1}_{k-1, k} = \begin{bmatrix} R_{k-1,k}^T & -R_{k-1,k}^Tt_{k-1,k} \\ 0 & 1 \end{bmatrix}$

4. Отже, оновлення матриць обертання та переміщення має вигляд:

    $T_{k,w} = \begin{bmatrix} R_{k-1, w} & t_{k-1, w} \\ 0 & 1 \end{bmatrix} \begin{bmatrix} R_{k-1,k}^T & -R_{k-1,k}^Tt_{k-1,k} \\ 0 & 1 \end{bmatrix} = \begin{bmatrix} R_{k, w} & t_{k, w} \\ 0 & 1 \end{bmatrix}$

    $R_{k,w}=R_{k-1,w}R^T_{k-1,k}$

    $t_{k,w}= -R_{k,w}t_{k-1,k} + t_{k-1, w}$ – переміщення із системи координат камери $k$ у $w$ відповідає положенню камери $k$ у системі координат $w$.